# SAM3 Test — VitroVision v2
Test Promptable Concept Segmentation (PCS) ด้วย text prompt

**Model:** `facebook/sam3` (ใช้ Sam3Model + Sam3Processor)
**Runtime:** ต้องเลือก GPU (Runtime → Change runtime type → T4 GPU)

> ⚠️ `facebook/sam3.1` ไม่มี Transformers integration (เป็น checkpoints ล้วน) — ใช้ `facebook/sam3` แทน

In [ ]:
# ติดตั้ง dependencies
!pip install -q transformers torch torchvision opencv-python pillow matplotlib numpy requests httpx

In [ ]:
# Login HF (ใส่ token ที่มี access facebook/sam3)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from transformers import Sam3Processor, Sam3Model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

print("กำลังโหลด SAM3...")
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("โหลดเสร็จ!")

In [ ]:
# อัปโหลดรูปจากเครื่อง
from google.colab import files
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
img = Image.open(img_path).convert("RGB")
print(f"รูป: {img_path} ({img.size})")

In [ ]:
prompts = ["leaf", "shoot", "plantlet", "stem", "plant tissue culture"]

fig, axes = plt.subplots(1, len(prompts), figsize=(5*len(prompts), 5))

for idx, prompt in enumerate(prompts):
    print(f">> prompt = \"{prompt}\"")

    inputs = processor(images=img, text=prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=0.5,
        mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist()
    )[0]

    masks = results["masks"]
    scores = results.get("scores", [])
    print(f"   found {len(masks)} masks")

    vis = np.array(img).copy()
    if len(masks) > 0:
        for i in range(len(masks)):
            mask_np = masks[i].cpu().numpy().astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis, contours, -1, (0, 255, 0), 2)

    axes[idx].imshow(vis)
    axes[idx].set_title(f"{prompt} ({len(masks)})")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# save result
for prompt in prompts:
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_instance_segmentation(
        outputs, threshold=0.5, mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist()
    )[0]

    vis = np.array(img).copy()
    for i in range(len(results["masks"])):
        mask_np = results["masks"][i].cpu().numpy().astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, contours, -1, (0, 255, 0), 2)
    Image.fromarray(vis).save(f"sam3_{prompt}.png")
    print(f"saved sam3_{prompt}.png")